
####Objetivo del notebook

Hallar las distancias entre clusters, las cuales se analizarán en notebook 03_5

Entrada:

    Tabla:  centroides_kmodes, cuyo origen es el notebook 03_2
            centroide_muerte , cuyo origen es el notebook 03_3

Salida: 

    Tabla:  distancias_centroides
            detalle_distancias_centroides


In [0]:
#Importar librerías

from itertools import combinations
import pandas as pd

In [0]:
#Leer tabla de centroides del modelo de 3 clusters

df_centroides_kmodes = spark.table(
    "ml_proyecto_7405607705157039.default.centroides_kmodes"
)

In [0]:
df_centroides_kmodes.display()

In [0]:
# Leer tabla del centroide del grupo muerte

df_centroide_muerte = spark.table(
    "ml_proyecto_7405607705157039.default.centroide_muerte"
)

In [0]:
df_centroide_muerte.display()

In [0]:
df_tablas_centroides_apilar = df_centroides_kmodes.unionByName(df_centroide_muerte)

In [0]:
df_tablas_centroides_apilar.display()

In [0]:
centroides = df_tablas_centroides_apilar.toPandas()

In [0]:
variables = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod"
]

In [0]:
def comparar_centroides(fila1, fila2, variables):

    iguales = []
    diferentes = []

    for variable in variables:

        if fila1[variable] == fila2[variable]:
            iguales.append(variable)
        else:
            diferentes.append(variable)

    return (
        len(diferentes),
        iguales,
        diferentes
    )

In [0]:
from itertools import combinations

resultados = []

for i, j in combinations(range(len(centroides)), 2):

    fila1 = centroides.iloc[i]
    fila2 = centroides.iloc[j]

    distancia, iguales, diferentes = comparar_centroides(
        fila1,
        fila2,
        variables
    )

    origen = fila1["cluster"]
    destino = fila2["cluster"]

    resultados.append({
        "origen": origen,
        "destino": destino,
        "distancia": distancia,
        "distancia_normalizada": round(distancia/len(variables),3),
        "variables_iguales": ", ".join(iguales),
        "variables_diferentes": ", ".join(diferentes)
    })

In [0]:
df_distancias = pd.DataFrame(resultados)

df_distancias

In [0]:
df_distancias_spark = spark.createDataFrame(df_distancias)

In [0]:
(
    df_distancias_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.distancias_centroides"
    )
)

In [0]:
spark.table(
    "ml_proyecto_7405607705157039.default.distancias_centroides"
).display()

In [0]:
detalle_diferencias = []

for i, j in combinations(range(len(centroides)), 2):

    fila1 = centroides.iloc[i]
    fila2 = centroides.iloc[j]

    for variable in variables:

        if fila1[variable] != fila2[variable]:

            detalle_diferencias.append({

                "origen": int(fila1["cluster"]),
                "destino": int(fila2["cluster"]),
                "variable": variable,
                "valor_origen": fila1[variable],
                "valor_destino": fila2[variable]

            })

df_detalle = pd.DataFrame(detalle_diferencias)

df_detalle

In [0]:
df_detalle_distancias_centroides_spark = spark.createDataFrame(df_detalle)

In [0]:
(
    df_detalle_distancias_centroides_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.detalle_distancias_centroides"
    )
)

In [0]:
spark.table(
    "ml_proyecto_7405607705157039.default.detalle_distancias_centroides"
).display()